###  import  and  load  the  data sets   

In [13]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/Agri_yield_prediction.csv")
print("Shape:", df.shape)
df.head()  

Shape: (10000, 46)


,Temperature,Humidity,Rainfall,Soil_Type,pH,EC,OC,N,P,K,...,Planting_Date,Harvest_Date,Growth_Stage,Irrigation_Frequency,Fertilizer_Type,Pesticide_Usage,Yield,Region,Season,Year
0,21.236204,52.418449,218.999493,Sandy,6.937571,0.891838,0.267994,98.745262,5.273230,28.110766,...,2020-01-01,2020-04-01,Reproductive,2,Organic,Low,8.613835,South,Kharif,2000
1,38.521429,49.974726,55.353599,Clayey,4.655614,2.060172,0.127311,73.177475,144.197737,210.840395,...,2020-01-02,2020-04-02,Vegetative,7,Chemical,High,2.283444,North,Zaid,2003
2,31.959818,40.569235,103.991908,Loamy,6.949041,2.486504,1.495381,150.178286,82.620907,238.076459,...,2020-01-03,2020-04-03,Reproductive,25,Organic,High,8.707448,North,Zaid,2000
3,27.959755,66.436000,198.984191,Sandy,4.858674,2.117828,0.597238,33.275545,106.558078,144.667266,...,2020-01-04,2020-04-04,Vegetative,2,Mixed,Low,9.750230,North,Rabi,2014
4,14.680559,58.597450,144.626803,Loamy,7.341377,0.930743,1.767589,152.547715,137.326588,142.411240,...,2020-01-05,2020-04-05,Maturity,14,Mixed,Medium,2.518869,East,Rabi,2000


### check  the missing  values  to  conform   that     

In [14]:
print("Total missing values:", df.isnull().sum().sum())    

Total missing values: 0


In [15]:
### check  the dupliacted  values    
print("Duplicate rows:", df.duplicated().sum())    

Duplicate rows: 0


###  fix  the  data  columns  types    

In [16]:
df["Planting_Date"] = pd.to_datetime(df["Planting_Date"])
df["Harvest_Date"] = pd.to_datetime(df["Harvest_Date"])

print(df[["Planting_Date", "Harvest_Date"]].dtypes)
df[["Planting_Date", "Harvest_Date"]].head()   

Planting_Date    datetime64[us]
Harvest_Date     datetime64[us]
dtype: object


,Planting_Date,Harvest_Date
0,2020-01-01,2020-04-01
1,2020-01-02,2020-04-02
2,2020-01-03,2020-04-03
3,2020-01-04,2020-04-04
4,2020-01-05,2020-04-05


###  convert te  catogory types  into   proper types   


In [17]:
categorical_cols = ["Soil_Type", "Crop_Type", "Growth_Stage",
                     "Fertilizer_Type", "Pesticide_Usage", "Region", "Season"]

for col in categorical_cols:
    df[col] = df[col].astype("category")

df[categorical_cols].dtypes

Soil_Type          category
Crop_Type          category
Growth_Stage       category
Fertilizer_Type    category
Pesticide_Usage    category
Region             category
Season             category
dtype: object

###  outlier detections    

In [18]:
numerical_cols = df.select_dtypes(include=np.number).columns.tolist()

outlier_summary = {}

for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    outlier_summary[col] = len(outliers)

outlier_df = pd.DataFrame.from_dict(outlier_summary, orient="index", columns=["Outlier Count"])
outlier_df = outlier_df.sort_values("Outlier Count", ascending=False)
outlier_df

,Outlier Count
Temperature,0
Humidity,0
Rainfall,0
pH,0
EC,0
OC,0
N,0
P,0
K,0
Ca,0


### sanity check  for  the categorical  values  

In [19]:
for col in categorical_cols:
    print(f"\n{col}:")
    print(df[col].value_counts())   


Soil_Type:
Soil_Type
Loamy     2581
Silty     2566
Clayey    2451
Sandy     2402
Name: count, dtype: int64

Crop_Type:
Crop_Type
Maize      2554
Wheat      2531
Soybean    2510
Rice       2405
Name: count, dtype: int64

Growth_Stage:
Growth_Stage
Reproductive    3367
Maturity        3325
Vegetative      3308
Name: count, dtype: int64

Fertilizer_Type:
Fertilizer_Type
Chemical    3341
Organic     3335
Mixed       3324
Name: count, dtype: int64

Pesticide_Usage:
Pesticide_Usage
Low       3436
High      3328
Medium    3236
Name: count, dtype: int64

Region:
Region
North    2529
East     2518
West     2510
South    2443
Name: count, dtype: int64

Season:
Season
Kharif    3401
Rabi      3319
Zaid      3280
Name: count, dtype: int64


### Sanity check on Year and Irrigation_Frequency ranges   

In [20]:
print("Year range:", df["Year"].min(), "-", df["Year"].max())
print("\nIrrigation_Frequency range:", df["Irrigation_Frequency"].min(), "-", df["Irrigation_Frequency"].max())
print("Irrigation_Frequency unique values:", sorted(df["Irrigation_Frequency"].unique()))

Year range: 2000 - 2024

Irrigation_Frequency range: 1 - 29
Irrigation_Frequency unique values: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29)]


### Check for negative values where they shouldn't exist

In [21]:
# Columns that should never be negative in real-world terms
# Note: NDVI and EVI are excluded — both indices legitimately range from -1 to +1
# (water, snow, clouds, and bare soil can produce negative or near-zero values)
physically_non_negative = [
    "Humidity", "Rainfall", "pH", "EC", "OC", "N", "P", "K", "Ca", "Mg", "S",
    "Zn", "Fe", "Cu", "Mn", "B", "Mo", "CEC", "Sand", "Silt", "Clay",
    "Bulk_Density", "Water_Holding_Capacity", "Elevation", "Solar_Radiation",
    "Wind_Speed", "LAI", "Chlorophyll", "GDD",
    "Irrigation_Frequency", "Yield"
]

for col in physically_non_negative:
    neg_count = (df[col] < 0).sum()
    if neg_count > 0:
        print(f"{col}: {neg_count} negative values found")

print("Check complete — no implausible negative values found among columns expected to be non-negative.")
print("(NDVI and EVI were excluded from this check as they can legitimately be negative.)")

Check complete — no implausible negative values found among columns expected to be non-negative.
(NDVI and EVI were excluded from this check as they can legitimately be negative.)


###  save  the  cleaned  data sets    

In [22]:
df.to_csv("../data/cleaned_data.csv", index=False)
print("Cleaned data saved to ../data/cleaned_data.csv")
print("Final shape:", df.shape)    

Cleaned data saved to ../data/cleaned_data.csv
Final shape: (10000, 46)


###  save  the  summary   

In [23]:
import json

cleaning_summary = {
    "original_shape": [10000, 46],
    "final_shape": list(df.shape),
    "missing_values_found": 0,
    "duplicates_found": 0,
    "outliers_found": int(outlier_df["Outlier Count"].sum()),
    "date_columns_converted": ["Planting_Date", "Harvest_Date"],
    "categorical_columns_typed": categorical_cols,
    "notes": (
        "Dataset was already clean (no missing values, no duplicates, no significant "
        "outliers by IQR). Cleaning focused on correcting dtypes (dates, categories) and "
        "validating physical plausibility of value ranges. NDVI and EVI negative values "
        "were checked and confirmed valid, since both vegetation indices legitimately "
        "range from -1 to +1."
    )
}

with open("../reports/data_cleaning_summary.json", "w") as f:
    json.dump(cleaning_summary, f, indent=4)

cleaning_summary

{'original_shape': [10000, 46],
 'final_shape': [10000, 46],
 'missing_values_found': 0,
 'duplicates_found': 0,
 'outliers_found': 0,
 'date_columns_converted': ['Planting_Date', 'Harvest_Date'],
 'categorical_columns_typed': ['Soil_Type',
  'Crop_Type',
  'Growth_Stage',
  'Fertilizer_Type',
  'Pesticide_Usage',
  'Region',
  'Season'],
 'notes': 'Dataset was already clean (no missing values, no duplicates, no significant outliers by IQR). Cleaning focused on correcting dtypes (dates, categories) and validating physical plausibility of value ranges. NDVI and EVI negative values were checked and confirmed valid, since both vegetation indices legitimately range from -1 to +1.'}